# 00 — Coleta de Dados
## Brasileirão Série A 2025 — Previsão de Rebaixamento

**Aluno:** Leonardo Feitosa | **UFPB — Ciência de Dados**

---

## Fonte dos Dados

Os dados foram coletados do site **Transfermarkt** (https://www.transfermarkt.com.br) utilizando as bibliotecas **Selenium** (para automação do navegador) e **BeautifulSoup** (para parsing do HTML), cobrindo todas as temporadas do Brasileirão Série A de **2014 a 2025**.

Para cada temporada foram extraídas informações sobre os 20 clubes participantes, incluindo:
- Tamanho do plantel
- Número de jogadores estrangeiros
- Idade média
- Valor de mercado médio e total

Os **pontos finais** de cada temporada (coluna `Pontos`) **não estão disponíveis no Transfermarkt** durante o andamento do campeonato. Por isso, eles foram adicionados manualmente via **PROCV no Excel**, cruzando os nomes dos clubes com uma planilha auxiliar de classificações finais de cada edição do campeonato. Para a temporada de **2025**, os pontos ainda não foram definidos, pois o campeonato está em andamento.

In [ ]:
# URLs utilizadas para scraping no Transfermarkt (uma por temporada)
urls_por_temporada = {
    2014: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2014',
    2015: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2015',
    2016: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2016',
    2017: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2017',
    2018: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2018',
    2019: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2019',
    2020: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2020',
    2021: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2021',
    2022: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2022',
    2023: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2023',
    2024: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2024',
    2025: 'https://www.transfermarkt.com.br/campeonato-brasileiro-serie-a/tabelle/wettbewerb/BRA1/saison_id/2025',
}

print(f'Total de temporadas coletadas: {len(urls_por_temporada)}')
print(f'Período: {min(urls_por_temporada.keys())} – {max(urls_por_temporada.keys())}')
print('\nURLs por temporada:')
for ano, url in urls_por_temporada.items():
    print(f'  {ano}: {url}')

## Estrutura dos Dados Coletados

Após a coleta e consolidação, o arquivo `BASE_FINAL.xlsx` (aba `CLUBES`) contém as seguintes colunas:

| Coluna | Tipo | Descrição |
|---|---|---|
| `Clube` | string | Nome do clube participante |
| `Plantel` | int | Número total de jogadores no plantel |
| `ø Idade` | float | Idade média dos jogadores do plantel |
| `Estrangeiros` | int | Número de jogadores estrangeiros no plantel |
| `ø Valor de Mercado` | float | Valor de mercado médio por jogador (em euros) |
| `Valor de Mercado Total` | float | Valor de mercado total do plantel (em euros) |
| `Temporada` | int | Ano da temporada (ex: 2014, 2015, …, 2025) |
| `Pontos` | int | Pontos finais no campeonato (NaN para 2025) |
| `Situacao` | string | Resultado da temporada: Rebaixado, Permaneceu, etc. |
| `Status` | string | Descrição adicional da situação do clube |

> **Nota:** As features utilizadas no modelo são `Plantel`, `Estrangeiros` e `Valor de Mercado Total`. A variável-alvo é derivada de `Situacao`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Carregamento dos dados
df = pd.read_excel(os.path.join('..', 'dados', 'BASE_FINAL.xlsx'), sheet_name='CLUBES')
df.columns = df.columns.str.strip()

print(f'Registros totais: {len(df)}')
print(f'Temporadas: {sorted(df["Temporada"].unique())}')
print(f'\nColunas: {list(df.columns)}')
df.head(10)

In [ ]:
# Informações gerais sobre tipos e valores nulos
df.info()

In [ ]:
# Criação da variável dependente binária
# 0 = Rebaixado | 1 = Permaneceu na Série A
df['Status_bin'] = df['Situacao'].apply(lambda x: 0 if str(x).strip().lower() == 'rebaixado' else 1)

print('Distribuição da variável dependente (Status_bin):')
print(df['Status_bin'].value_counts().rename({0: 'Rebaixado (0)', 1: 'Permaneceu (1)'}))

# Gráfico de pizza — apenas temporadas com resultado definido (2014–2024)
fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(
    [44, 196],
    labels=['Rebaixado\n(0)', 'Permaneceu\n(1)'],
    autopct='%1.1f%%',
    colors=['#e74c3c', '#2ecc71'],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
ax.set_title('Distribuição da Variável Dependente\n(2014–2024)', fontweight='bold')
plt.tight_layout()
plt.show()

## Como os Pontos Foram Adicionados

Os **pontos finais** de cada temporada (coluna `Pontos`) não estão disponíveis no Transfermarkt durante o andamento do campeonato — a plataforma foca em dados financeiros e de elenco, não em desempenho esportivo detalhado.

Para contornar isso, os pontos foram adicionados **manualmente via PROCV no Excel**: após o término de cada campeonato, a classificação final era importada de uma planilha auxiliar (com os pontos de cada clube por temporada) e cruzada com a base principal pelo nome do clube, utilizando a função `PROCV` (equivalente ao `VLOOKUP` em inglês).

Para a temporada de **2025**, a coluna `Pontos` permanece vazia (`NaN`), pois o campeonato ainda está em disputa. O modelo de previsão utiliza apenas as features de plantel e valor de mercado, que estão disponíveis **antes do início** da temporada.

In [ ]:
# Distribuição de pontos por situação final (excluindo 2025, sem dados)
df_hist = df[df['Temporada'] < 2025].copy()

fig, ax = plt.subplots(figsize=(10, 5))

for status, label, cor in [(0, 'Rebaixado', '#e74c3c'), (1, 'Permaneceu', '#2ecc71')]:
    dados = df_hist[df_hist['Status_bin'] == status]['Pontos'].dropna()
    ax.hist(dados, bins=15, alpha=0.6, label=label, color=cor, edgecolor='white')

ax.axvline(x=45, color='gray', linestyle='--', alpha=0.7, label='~45 pts (zona de perigo)')
ax.set_xlabel('Pontos Finais')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de Pontos por Situação Final (2014–2024)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## Conclusão da Coleta

A base de dados está consolidada e pronta para análise. Resumo:

- **240 registros** no total (20 clubes × 12 temporadas: 2014–2025)
- **220 registros com rótulo definido** (2014–2024, utilizados em treino/teste)
- **20 registros de previsão** (temporada 2025)
- **3 features** selecionadas para o modelo: `Plantel`, `Estrangeiros`, `Valor de Mercado Total`
- **1 variável-alvo** binária: `Status_bin` (0 = Rebaixado, 1 = Permaneceu)

Os dados estão prontos para a etapa de **Análise Exploratória** (`01_analise_exploratoria.ipynb`).